# 01 Data audit
**Question.** Which datasets can we legitimately use, what do they support, and what can they NOT support?

Findings up front: no open dataset carries APP labels *with* device/session/beneficiary-registration signals, so PRESAGE uses (a) three real public datasets as independent benchmarks (not APP-labelled) and (b) a clearly synthetic APP/mule world (`sim_v1`). Dataset cards live in `research/dataset_cards/`.

In [1]:
import sys, pathlib, json, warnings; warnings.filterwarnings('ignore')
ROOT = pathlib.Path.cwd().parent if pathlib.Path.cwd().name == 'notebooks' else pathlib.Path.cwd()
sys.path.insert(0, str(ROOT / 'src'))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
plt.rcParams.update({'figure.dpi': 100, 'axes.spines.top': False, 'axes.spines.right': False})
def table(name):
    p = ROOT / 'reports' / 'tables' / name
    return pd.read_csv(p) if p.exists() else None

## Real datasets (downloaded by `scripts/download_data.py`)

In [2]:
raw = ROOT / 'data' / 'raw'
print({p.name: f'{p.stat().st_size/1e6:.0f} MB' for p in raw.glob('*') if p.is_file()})
ps = pd.read_csv(raw / 'paysim.csv', usecols=['step','type','amount','isFraud'])
print('PaySim rows', len(ps), 'fraud rate %.4f%%' % (100*ps.isFraud.mean()), 'hours', ps.step.max())
print(ps.groupby('type').agg(rows=('amount','size'), fraud=('isFraud','sum'), mean_amount=('amount','mean')).round(1))

{'paysim.csv': '494 MB', 'elliptic_txs_edgelist.csv': '4 MB', 'HI-Small_Trans.csv': '476 MB', 'elliptic_txs_features.csv': '690 MB', 'elliptic_txs_classes.csv': '3 MB'}


PaySim rows 6362620 fraud rate 0.1291% hours 743


             rows  fraud  mean_amount
type                                 
CASH_IN   1399284      0     168920.2
CASH_OUT  2237500   4116     176274.0
DEBIT       41432      0       5483.7
PAYMENT   2151495      0      13057.6
TRANSFER   532909   4097     910647.0


In [3]:
ibm = pd.read_csv(raw / 'HI-Small_Trans.csv', usecols=['Timestamp','Payment Format','Is Laundering','From Bank'])
print('IBM AML HI-Small rows', len(ibm), 'laundering %.4f%%' % (100*ibm['Is Laundering'].mean()), ibm.Timestamp.min(), '->', ibm.Timestamp.max())
print(ibm.groupby('Payment Format')['Is Laundering'].agg(['size','sum','mean']).round(5))
del ibm

IBM AML HI-Small rows 5078345 laundering 0.1019% 2022/09/01 00:00 -> 2022/09/18 16:18


                   size   sum     mean
Payment Format                        
ACH              600797  4483  0.00746
Bitcoin          146091    56  0.00038
Cash             490891   108  0.00022
Cheque          1864331   324  0.00017
Credit Card     1323324   206  0.00016
Reinvestment     481056     0  0.00000
Wire             171855     0  0.00000


In [4]:
cl = pd.read_csv(raw / 'elliptic_txs_classes.csv'); print(cl['class'].value_counts())
ed = pd.read_csv(raw / 'elliptic_txs_edgelist.csv'); print('Elliptic edges', len(ed))

class
unknown    157205
2           42019
1            4545
Name: count, dtype: int64
Elliptic edges 234355


## What each dataset supports
| Dataset | txn history | graph | temporal | AML/mule-like | APP labels | device/session |
|---|---|---|---|---|---|---|
| PaySim | yes | sender-recipient | 743 hourly steps | no | **no** | no |
| IBM AML HI-Small | yes | account graph, cross-bank | 18 days | **laundering networks** | **no** | no |
| Elliptic | graph of transactions | yes | 49 steps | illicit flows | **no** | no |
| sim_v1 (synthetic) | yes | yes + device/ip | 120 days | **mule networks** | **yes (synthetic)** | yes (synthetic) |

Rule followed throughout: results on PaySim/IBM/Elliptic are reported as transaction-fraud/AML benchmarks, never as APP fraud.

In [5]:
x = table('real_datasets.csv'); print(x.round(3).to_string(index=False) if x is not None else 'run scripts/run_real_datasets.py')

                             dataset                       feature_set  pr_auc  pr_auc_lo  pr_auc_hi  roc_auc  recall  precision    f1    f2   mcc   fpr  test_rows  test_positives  test_prevalence
                              PaySim                          txn_only   0.987      0.980      0.995    0.999   0.986      0.935 0.960 0.975 0.958 0.003      15283             579            0.038
                              PaySim         txn+behaviour+beneficiary   0.989      0.983      0.996    1.000   0.991      0.933 0.961 0.979 0.960 0.003      15283             579            0.038
                              PaySim   txn+behaviour+beneficiary+graph   0.988      0.981      0.994    0.999   0.984      0.936 0.960 0.974 0.958 0.003      15283             579            0.038
IBM-AML-HI-Small(enriched subsample)                          txn_only   0.631      0.547      0.752    0.488   0.377      0.690 0.487 0.414 0.115 0.265        126              77            0.611
IBM-AML-HI-Smal